# SBERT Sentence Analysis

## 1. Preparations

### 1.1 Read Data
This new data is obtained after some further anonymization, and by putting previously incorrectly split sentences together, and removed duplicates

In [1]:
import pandas as pd
# read the sentence data 
df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx")
# check the head
df.head()

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,1,I was rushed into the [PRESSION] two weeks ago...
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",2,I was acutely under a painful pressure on the ...
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,3,It started 1 hour after dinner.
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...",4,"I had migraines all day, so I hadn't eaten much."
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...",5,"I got very nauseous, vomiting, dizzy and rejuv..."


In [2]:
# clean sentences as inputs
sentences = df["Clean_Sentence"].to_list()

In [3]:
# print the size of the data
len(sentences)
# There are 41143 sentences 

41143

### 1.2. Import the list of stopwords

In [4]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/persistent/mijnidbcoachnlp/data/analysis_data/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg", "coach", "mijnibdcoach", "dr", "uur", "dgs"
]

dutch_stopwords.extend(extra_list)

### 1.4. Embed the lists of sentences with sentence-transformer multilingual-v1

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

# load the embedding model
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")

# run the following for embeddings if no pre-saved embeddings can be loaded
#embeddings = embedding_model.encode(sentences, show_progress_bar=True)
# save the pre-calculated embeddings 
#np.save('/workspace/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy', embeddings)

/workspace/persistent/mijnidbcoachnlp/venv/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
# load pre-saved embeddings if you have, otherwise calculate them using the commented-out codes
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")
embeddings = np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy")

In [7]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Create a dictionary representation of the words in topics
tokenized_texts = [doc.split() for doc in sentences]  # Tokenizing by splitting on spaces
dictionary = Dictionary(tokenized_texts)

def get_top_words(topic_model):
    topics = topic_model.get_topics()
    top_words = [[word for word, _ in topic[:10]] for topic in topics.values() if topic]  # Top 10 words per topic
    return top_words

def calculate_c_v(topic_model):
    top_words = get_top_words(topic_model)
    coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')
    coherence_score = coherence_model.get_coherence()
    return coherence_score

In [8]:
# disable parallelism to avoid some warnings
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 2. Fine-tune the ST model

In [15]:
import random
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# Precompute embeddings (this part depends on your chosen embedding model)
# Example: Assuming 'embeddings' is already computed and available
# embeddings = embedding_model.encode(sentences)

# Initialize BERTopic with the pre-fit vectorizer
topic_model = BERTopic()

# Define parameter ranges
range_n_neighbors = [10, 20, 30, 40, 50]
range_n_components = [2, 3, 4, 5, 6]
range_min_dist = [0.0, 0.05, 0.1]

# Number of random configurations to generate
num_configs = 3  # Adjust this as needed

# Generate random UMAP configurations
umap_configs = [
    {
        "n_neighbors": random.choice(range_n_neighbors),
        "n_components": random.choice(range_n_components),
        "min_dist": random.choice(range_min_dist),
    }
    for _ in range(num_configs)
]

# Define the ranges for HDBSCAN configurations
range_min_cluster_size = [10, 20, 30]  # Example values
range_min_samples = [10, 20, 30]  # Example values

# Generate random HDBSCAN configurations
hdbscan_configs = [
    {
        "min_cluster_size": random.choice(range_min_cluster_size),
        "min_samples": random.choice([x for x in range_min_samples if x <= random.choice(range_min_cluster_size)]),
    }
    for _ in range(num_configs)
]

# Initialize a dictionary to keep track of configurations and their associated coherence scores
tried_configs = {}

# Optionally, load previously tried configurations from a file (if running multiple times)
try:
    with open("tried_configs_st.pkl", "rb") as f:
        tried_configs = pickle.load(f)
except FileNotFoundError:
    pass  # If no previous file exists, we start fresh

# Variables to track the best model
best_topic_model = None
best_coherence_score = -float('inf')  # Initializing to a very low score
best_configure = None

# Set the outlier thresholds (upper and lower)
upper_outlier_threshold = 0.5  # Maximum allowed outlier percentage (5% of points)
lower_outlier_threshold = 0.2  # Minimum allowed outlier percentage (2% of points)

In [17]:
# Fine-tuning loop
for umap_params in umap_configs:
    for hdbscan_params in hdbscan_configs:
        config_tuple = (frozenset(umap_params.items()), frozenset(hdbscan_params.items()))

        # Skip if the configuration has already been tried
        if config_tuple in tried_configs:
            print(f"Skipping already tried configuration: {umap_params}, {hdbscan_params}")
            continue

        # Initialize UMAP and HDBSCAN with current configurations
        umap_model = UMAP(**umap_params)
        hdbscan_model = HDBSCAN(**hdbscan_params)
        print(f"Fitting model for UMAP {umap_params} and HDBSCAN {hdbscan_params}")

        # Update BERTopic with the new UMAP and HDBSCAN models
        topic_model.umap_model = umap_model
        topic_model.hdbscan_model = hdbscan_model
        topic_model.vectorizer_model = CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b')
        topic_model.calculate_probabilities = False

        # Fit the model
        topics, probs = topic_model.fit_transform(sentences, embeddings)

        # Calculate coherence score
        current_coherence_score = calculate_c_v(topic_model)

        # Check if the model has at least 20 topics
        num_topics = len(topic_model.get_topics())
        if num_topics < 20:
            print(f"Skipping model with {num_topics} topics (less than 20).")
            tried_configs[config_tuple] = "skipped"
            continue

        # Check outlier percentage
        outlier_cluster_size = sum(1 for label in hdbscan_model.labels_ if label == -1)
        outlier_percentage = outlier_cluster_size / len(hdbscan_model.labels_)
        
        if outlier_percentage > upper_outlier_threshold:
            print(f"Skipping model with {outlier_percentage * 100}% outliers (above {upper_outlier_threshold * 100}%).")
            tried_configs[config_tuple] = "skipped"
            continue
        elif outlier_percentage < lower_outlier_threshold:
            print(f"Skipping model with {outlier_percentage * 100}% outliers (below {lower_outlier_threshold * 100}%).")
            tried_configs[config_tuple] = "skipped"
            continue
        
        # Store the valid configuration and score
        tried_configs[config_tuple] = current_coherence_score

        # Update best model
        if current_coherence_score > best_coherence_score:
            best_coherence_score = current_coherence_score
            best_topic_model = topic_model
            best_configure = (umap_params, hdbscan_params)

        print(f"Coherence score: {current_coherence_score}\n")

        # Save tried configurations after every iteration
        with open("tried_configs_st.pkl", "wb") as f:
            pickle.dump(tried_configs, f)


Fitting model for UMAP {'n_neighbors': 50, 'n_components': 4, 'min_dist': 0.05} and HDBSCAN {'min_cluster_size': 10, 'min_samples': 10}
Coherence score: 0.3662643130528934

Fitting model for UMAP {'n_neighbors': 50, 'n_components': 4, 'min_dist': 0.05} and HDBSCAN {'min_cluster_size': 10, 'min_samples': 20}
Coherence score: 0.35050085496611455

Fitting model for UMAP {'n_neighbors': 50, 'n_components': 4, 'min_dist': 0.05} and HDBSCAN {'min_cluster_size': 30, 'min_samples': 10}
Coherence score: 0.3582903721014854

Fitting model for UMAP {'n_neighbors': 20, 'n_components': 4, 'min_dist': 0.1} and HDBSCAN {'min_cluster_size': 10, 'min_samples': 10}
Coherence score: 0.3666985394181407

Fitting model for UMAP {'n_neighbors': 20, 'n_components': 4, 'min_dist': 0.1} and HDBSCAN {'min_cluster_size': 10, 'min_samples': 20}
Coherence score: 0.36657597532332886

Fitting model for UMAP {'n_neighbors': 20, 'n_components': 4, 'min_dist': 0.1} and HDBSCAN {'min_cluster_size': 30, 'min_samples': 10}


In [18]:
# After the loop, print the best configurations and score
print("Best UMAP and HDBSCAN configurations based on coherence score:")
print(f"Best Coherence Score: {best_coherence_score}")
print(f"Best UMAP Params: {best_configure[0]}")
print(f"Best HDBSCAN Params: {best_configure[1]}")

Best UMAP and HDBSCAN configurations based on coherence score:
Best Coherence Score: 0.368752327810459
Best UMAP Params: {'n_neighbors': 20, 'n_components': 3, 'min_dist': 0.05}
Best HDBSCAN Params: {'min_cluster_size': 10, 'min_samples': 10}


In [20]:
import pickle

# Load the file
with open("tried_configs_st.pkl", "rb") as f:
    tried_configs = pickle.load(f)

# Print the contents
for config, score in tried_configs.items():
    print(f"Config: {config}, Score: {score}")


Config: (frozenset({('n_neighbors', 10), ('min_dist', 0.1), ('n_components', 5)}), frozenset({('min_cluster_size', 30), ('min_samples', 10)})), Score: 0.3425443099837642
Config: (frozenset({('n_neighbors', 10), ('n_components', 4), ('min_dist', 0.0)}), frozenset({('min_samples', 30), ('min_cluster_size', 20)})), Score: 0.35067638061555395
Config: (frozenset({('n_neighbors', 10), ('n_components', 4), ('min_dist', 0.0)}), frozenset({('min_cluster_size', 30), ('min_samples', 10)})), Score: 0.3362990734428644
Config: (frozenset({('n_neighbors', 10), ('n_components', 4), ('min_dist', 0.0)}), frozenset({('min_samples', 30), ('min_cluster_size', 10)})), Score: 0.3490907784930923
Config: (frozenset({('n_neighbors', 10), ('n_components', 3), ('min_dist', 0.0)}), frozenset({('min_samples', 30), ('min_cluster_size', 20)})), Score: 0.33991947203180106
Config: (frozenset({('n_neighbors', 10), ('n_components', 3), ('min_dist', 0.0)}), frozenset({('min_cluster_size', 30), ('min_samples', 10)})), Scor

In [23]:
best_topic_model.save("/workspace/persistent/mijnidbcoachnlp/saved_models/best_cv_score_st_model", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [24]:
calculate_c_v(best_topic_model)

0.3552550163346731

In [21]:
best_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,18597,-1_afspraak_vraag_gaat_weet,"[afspraak, vraag, gaat, weet, volgende, goed, ...","[en/of , Het gaat de laatste 2 weken niet zo g..."
1,0,3367,0_bloed_prikken_bloedprikken_ontlasting,"[bloed, prikken, bloedprikken, ontlasting, sli...",[Afgelopen vrijdag ben ik bloed laten prikken....
2,1,1291,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, cont...","[Ja bij mijn huisarts., Ja bij mijn huisarts.,..."
3,2,556,2_medicatie_apotheek_medicijnen_medicijn,"[medicatie, apotheek, medicijnen, medicijn, re...","[Nog steeds geen recept binnen bij apotheek!, ..."
4,3,516,3_pijn_buikpijn_pijnlijk_buik,"[pijn, buikpijn, pijnlijk, buik, pijnlijke, ru...","[Dit is ondoenlijk van wege de pijn!, Maar ik ..."
...,...,...,...,...,...
242,241,11,241_doorgestuurd_apotheekderrein_zorginstellin...,"[doorgestuurd, apotheekderrein, zorginstelling...",[Het recept van de picoprep zou zijn doorgestu...
243,242,11,242_optie_rele_alternatieve_buitenland,"[optie, rele, alternatieve, buitenland, vastge...","[Zijn er nog andere optie?, Is dit een reële o..."
244,243,10,243_verwachtte_kim_beloofd_schrijft,"[verwachtte, kim, beloofd, schrijft, weekend, ...",[Ik had beloofd nog even te laten weten hoe he...
245,244,10,244_contactmoment_emailen_vlg_simpony,"[contactmoment, emailen, vlg, simpony, kontakt...",[ik heb vorige week via de mail nieuwe simpon...


In [25]:
# check if reduce outliers now work
# Reduce outliers
topics = best_topic_model.topics_
new_topics = best_topic_model.reduce_outliers(sentences, topics)

In [26]:
best_topic_model.update_topics(sentences, topics=new_topics, vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b'))

2025-02-13 05:22:51,663 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [27]:
# check topic_info after outlier reduction
best_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_heermevrouw_aangenaam,"[verstandigst, trant, heermevrouw, aangenaam, ...","[en/of , Het gaat de laatste 2 weken niet zo g..."
1,0,3567,0_bloed_prikken_ontlasting_bloedprikken,"[bloed, prikken, ontlasting, bloedprikken, sli...",[Afgelopen vrijdag ben ik bloed laten prikken....
2,1,1458,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, cont...","[Ja bij mijn huisarts., Ja bij mijn huisarts.,..."
3,2,650,2_medicatie_medicijn_apotheek_medicijnen,"[medicatie, medicijn, apotheek, medicijnen, vo...","[Nog steeds geen recept binnen bij apotheek!, ..."
4,3,694,3_pijn_buikpijn_pijnlijk_buik,"[pijn, buikpijn, pijnlijk, buik, pijnlijke, ru...","[Dit is ondoenlijk van wege de pijn!, Maar ik ..."
...,...,...,...,...,...
242,241,36,241_doorgestuurd_picoprep_gestuurd_doorgezet,"[doorgestuurd, picoprep, gestuurd, doorgezet, ...",[Het recept van de picoprep zou zijn doorgestu...
243,242,75,242_optie_alternatief_momenten_mogelijkheden,"[optie, alternatief, momenten, mogelijkheden, ...","[Zijn er nog andere optie?, Is dit een reële o..."
244,243,61,243_nou_wekelijks_geplande_volgen,"[nou, wekelijks, geplande, volgen, geval, inna...",[Ik had beloofd nog even te laten weten hoe he...
245,244,48,244_gelezen_laboratorium_buikgriep_afgesproken,"[gelezen, laboratorium, buikgriep, afgesproken...",[ik heb vorige week via de mail nieuwe simpon...


In [28]:
best_topic_model.save("/workspace/persistent/mijnidbcoachnlp/saved_models/best_cv_score_st_model_ro", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [29]:
calculate_c_v(best_topic_model)

0.3566552534427884

# map the topics to one of the two labels

In [103]:
from bertopic import BERTopic
topic_model = BERTopic.load("/workspace/persistent/mijnidbcoachnlp/saved_models/best_cv_score_st_model_ro", embedding_model=embedding_model)

/workspace/persistent/mijnidbcoachnlp/venv/lib/python3.8/site-packages/bertopic/_save_utils.py:187: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.

/workspace/persistent/mijni

In [24]:
topic_model.get_topic_info().head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_heermevrouw_aangenaam,"[verstandigst, trant, heermevrouw, aangenaam, ...",NaN
1,0,3567,0_bloed_prikken_ontlasting_bloedprikken,"[bloed, prikken, ontlasting, bloedprikken, sli...",NaN
2,1,1458,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, cont...",NaN
3,2,650,2_medicatie_medicijn_apotheek_medicijnen,"[medicatie, medicijn, apotheek, medicijnen, vo...",NaN
4,3,694,3_pijn_buikpijn_pijnlijk_buik,"[pijn, buikpijn, pijnlijk, buik, pijnlijke, ru...",NaN


In [105]:
# settings for plotly
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default = "browser" # option to open the plots in brower

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
#topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 245/245 [00:00<00:00, 353.42it/s]


In [106]:
# Reduce the topics with a similarity threshold < 0.5
threshold = 0.5 # Define the threshold for merging
filtered_hierarchical_topics = hierarchical_topics[hierarchical_topics["Distance"] <= threshold]

# Ensure the loop stops when no more topics can be merged
while not filtered_hierarchical_topics.empty:  

    list_topics_to_merge = filtered_hierarchical_topics["Topics"].to_list()

    # Merge topics
    topic_model.merge_topics(sentences, list_topics_to_merge)

    # Recompute hierarchical topics after merging
    hierarchical_topics = topic_model.hierarchical_topics(sentences)

    # Update the filtered topics for the next iteration
    filtered_hierarchical_topics = hierarchical_topics[hierarchical_topics["Distance"] <= threshold]


100%|██████████| 208/208 [00:00<00:00, 360.28it/s]


In [109]:

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 208/208 [00:00<00:00, 361.57it/s]


In [108]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_aangenaam_zenuwachtig,"[verstandigst, trant, aangenaam, zenuwachtig, ...","[Of iets in die trant., Wat is voor mij het ve..."
1,0,3567,0_bloed_prikken_ontlasting_bloedprikken,"[bloed, prikken, ontlasting, bloedprikken, sli...",[Maar moet ik niet ook nog bloed laten prikken...
2,1,1458,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, cont...","[Volgens de huisarts heb ik ze gehad in '89., ..."
3,2,1049,2_recept_nieuw_apotheek_sturen,"[recept, nieuw, apotheek, sturen, herhaalrecep...",[Zou u een nieuw recept naar de apotheek in he...
4,3,819,3_pijn_buikpijn_buik_pijnlijk,"[pijn, buikpijn, buik, pijnlijk, rug, pijnlijk...","[Dit is ondoenlijk van wege de pijn!, Pijn is ..."
...,...,...,...,...,...
205,204,35,204_eerlijk_geboorte_gezegd_hoor,"[eerlijk, geboorte, gezegd, hoor, tips, graag,...","[Ik neem aan dat ze daar eerlijk over is., Ik ..."
206,205,34,205_geboortedatum_geboorte_geb_geboren,"[geboortedatum, geboorte, geb, geboren, hpm, a...","[De geboortedatum is ., geboortedatum is: ., M..."
207,206,31,206_nieuws_goed_goede_geloven,"[nieuws, goed, goede, geloven, super, verwacht...","[dat is goed nieuws., Dat is heel goed nieuws!..."
208,207,30,207_ggd_hoor_tel_graag,"[ggd, hoor, tel, graag, eem, secretaresse, uit...","[Graag hoor ik van u, Graag hoor ik van jullie..."


In [87]:
topics_to_merge_manually = [
    
]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,259,-1_verstandigst_trant_aangenaam_zenuwachtig,"[verstandigst, trant, aangenaam, zenuwachtig, ...","[Of iets in die trant., Wat is voor mij het ve..."
1,0,3567,0_bloed_prikken_ontlasting_slijm,"[bloed, prikken, ontlasting, slijm, bloedprikk...",[Afgelopen maandag 4 juli heb ik bloed laten p...
2,1,2996,1_recept_apotheek_nieuw_medicatie,"[recept, apotheek, nieuw, medicatie, tabletten...","[Graag recept naar Apotheek ., Zouden jullie e..."
3,2,2640,2_pijn_voel_gevoel_buikpijn,"[pijn, voel, gevoel, buikpijn, last, buik, voe...","[Pijn is al wat minder., Nergens meer last van..."
4,3,1610,3_afspraak_gepland_sessie_poli,"[afspraak, gepland, sessie, poli, staan, staat...","[Dit genoeg tot de afspraak., Ik heb giser een..."
...,...,...,...,...,...
162,161,37,161_tegemoet_zie_reactie_hieronder,"[tegemoet, zie, reactie, hieronder, kliniek, g...","[Graag zie ik u reactie tegemoet., Zie u react..."
163,162,36,162_doorgestuurd_picoprep_gestuurd_zorginstelling,"[doorgestuurd, picoprep, gestuurd, zorginstell...",[Dit mag doorgestuurd worden naar de [APOTHEEK...
164,163,34,163_geboortedatum_geboorte_geb_geboren,"[geboortedatum, geboorte, geb, geboren, hpm, a...","[Geboortedatum is, De geboortedatum is ., De g..."
165,164,31,164_nieuws_goed_goede_geloven,"[nieuws, goed, goede, geloven, super, verwacht...","[Inderdaad goed nieuws., Dat is goed nieuws., ..."


In [72]:
calculate_c_v(topic_model)

0.3686722770522602

In [ ]:
# import the large annotated set
annotated_df_new = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/new_annotated_df_lg.xlsx", index_col=0)
# check the head of lg set
annotated_df_lg = annotated_df_lg.loc[:, ~annotated_df_lg.columns.str.contains('^Unnamed')]
annotated_df_lg.head()

In [ ]:
annotated_df_lg['Sentence_ID'] = annotated_df_lg['Sentence_ID'].astype(str)

In [ ]:
# reduce out